# W7｜Debug：偏差最早发生在哪一步？

程序能跑完，结果却不对。先用固定输入复现，再决定在哪里停、要看哪个值。修复后用原输入和另一份输入重新检查。

## 本周路线

1. **看数据有没有走丢**：有效的 `0` 被当成缺失值；校正函数拿错了参数。
2. **追踪循环和更新**：漏了最后一个窗口；迟到的旧记录覆盖了新记录。
3. **分开检查多个环节**：回溯试点走错起点；组内统计与整体统计有两处独立故障。

## 如何使用

1. 在 VS Code 选择 `week-07/.venv` 的 Python kernel。每题先读需求，运行输入格，再正常运行 Playground，记下实际输出。起始断言失败是本周的练习材料。
2. 在 Playground 左侧点一个断点，从运行按钮旁的菜单选择 **Debug Cell**。暂停后在 **Variables** 看当前变量，必要时用 **Step Over** 走过一行、用 **Step Into** 进入函数、用 **Call Stack** 查看调用者。用 **Continue** 继续运行。
3. 在 Playground 修改并重跑。把修好的实现填到「函数组装」，再运行「单元检查」。组装格起初只有函数壳。参考解答折叠在每题末尾。

**同桌检查**：用自己的 Notebook 轮流展示输入、停点与实际看到的值、改动、同一输入重跑后的输出。还没修完就展示已经查过的一步，并指出下一步要看的值。

## 场景一｜有效的 `0` 怎么消失了？

### 场景与需求

一串仪器读数保存在列表中。`None` 代表仪器没有读数，`0` 是一次有效观测。平均值只使用有效观测。如果没有有效观测，返回 `None`。

### 输入与输出

- `[0, None, 6]` → `3`，两次有效观测为 `0` 和 `6`。
- `[0, 0]` → `0`；`[None, None]` → `None`。

### 你的任务

实现 `mean_observed(values)`，接收读数列表并返回均值或 `None`。先区分「有效观测在筛选时丢了」与「观测都在，均值却算错了」。挑一个能看到筛选结果的位置停住。

In [ ]:
q1_values = [0, None, 6]
q1_expected = 3
print("读数", q1_values, "预期均值", q1_expected)

In [ ]:
def mean_observed_trial(values):
    observed = []
    for value in values:
        if value:
            observed.append(value)
    if not observed:
        return None
    return sum(observed) / len(observed)


q1_actual = mean_observed_trial(q1_values)
print("实际均值", q1_actual)
assert q1_actual == q1_expected, "均值应计入有效的 0"

In [ ]:
def mean_observed(values):
    raise NotImplementedError("填入调试后的实现")

In [ ]:
assert mean_observed([0, None, 6]) == 3
assert mean_observed([0, 0]) == 0
assert mean_observed([None, None]) is None
assert mean_observed([]) is None
assert mean_observed([-4, None, 2]) == -1
print("场景一：全部检查通过")

<details>
<summary>参考解答（点击展开）</summary>

```python
def mean_observed(values):
    observed = []
    for value in values:
        if value is not None:
            observed.append(value)
    if not observed:
        return None
    return sum(observed) / len(observed)
```

#### 错误原因

在筛选循环结束后停住，`observed` 只有 `[6]`。`if value` 把 `0` 和 `None` 一起排除了。

#### 修正前后的本质差别

判断条件从「值为真」变为「值不是 `None`」。均值公式保持一致，送进去的观测改为 `[0, 6]`。

</details>

## 场景二｜校正函数单测通过，报告怎么还是算反了？

### 场景与需求

仪器原始读数带有固定偏移。校正值等于「原始读数 − 偏移」。报告应保留原始读数、偏移和校正值，方便回查。

### 输入与输出

- 原始读数 `12`、偏移 `3` → `{"raw": 12, "offset": 3, "corrected": 9}`。
- 原始读数 `4`、偏移 `7` → `{"raw": 4, "offset": 7, "corrected": -3}`。

### 你的任务

`correct_reading(raw, offset)` 已经有单独的检查。修复 `build_calibration_report(raw, offset)`，让它返回完整报告。把断点打在校正函数内，两次停住时分别查看它收到的值。第二次停住后，从调用栈找到参数是从哪一层传来的。

In [ ]:
q2_raw = 12
q2_offset = 3
q2_expected = {"raw": 12, "offset": 3, "corrected": 9}
print("预期报告", q2_expected)

In [ ]:
def correct_reading_trial(raw, offset):
    return raw - offset


def build_calibration_report_trial(raw, offset):
    corrected = correct_reading_trial(offset, raw)
    return {"raw": raw, "offset": offset, "corrected": corrected}


assert correct_reading_trial(12, 3) == 9
print("直接调用校正函数通过")
q2_actual = build_calibration_report_trial(q2_raw, q2_offset)
print("实际报告", q2_actual)
assert q2_actual == q2_expected, "报告中的校正值有误"

In [ ]:
def correct_reading(raw, offset):
    return raw - offset


def build_calibration_report(raw, offset):
    # 用上面的 correct_reading 做校正。
    raise NotImplementedError("填入调试后的实现")

In [ ]:
assert correct_reading(12, 3) == 9
assert correct_reading(4, 7) == -3
assert build_calibration_report(12, 3) == {"raw": 12, "offset": 3, "corrected": 9}
assert build_calibration_report(4, 7) == {"raw": 4, "offset": 7, "corrected": -3}
assert build_calibration_report(4, -2) == {"raw": 4, "offset": -2, "corrected": 6}
print("场景二：全部检查通过")

<details>
<summary>参考解答（点击展开）</summary>

```python
def correct_reading(raw, offset):
    return raw - offset


def build_calibration_report(raw, offset):
    corrected = correct_reading(raw, offset)
    return {"raw": raw, "offset": offset, "corrected": corrected}
```

#### 错误原因

报告调用的那一轮，校正函数收到 `raw = 3`、`offset = 12`。切回调用栈上一层，可以看到报告函数把实参顺序写反了。

#### 修正前后的本质差别

校正仍是减法。修正发生在调用处，原始读数和偏移分别传给对应形参。

</details>

## 场景三｜最后一个窗口去哪了？

### 场景与需求

取一段连续读数，计算它们的和，再向后移动一个位置。只计算长度完整的窗口，按起点从小到大返回结果。窗口长度是正整数。

### 输入与输出

- `[2, 7, 4, 1]`、宽度 `2` → `[9, 11, 5]`，三个窗口是 `[2, 7]`、`[7, 4]`、`[4, 1]`。
- 同一列表、宽度 `4` → `[14]`；宽度 `1` → `[2, 7, 4, 1]`；宽度 `5` → `[]`。

### 你的任务

修复 `window_sums(values, width)`，接收列表和窗口宽度，返回各窗口的和。先判断最后一个窗口是算了但没收进列表，还是程序从未到过它的起点。要观察哪个变量？

In [ ]:
q3_values = [2, 7, 4, 1]
q3_width = 2
q3_expected = [9, 11, 5]
print("读数", q3_values, "宽度", q3_width, "预期", q3_expected)

In [ ]:
def window_sums_trial(values, width):
    totals = []
    for start in range(len(values) - width):
        window = values[start : start + width]
        totals.append(sum(window))
    return totals


q3_actual = window_sums_trial(q3_values, q3_width)
print("实际窗口和", q3_actual)
assert q3_actual == q3_expected, "每个完整窗口都应被计算"

In [ ]:
def window_sums(values, width):
    raise NotImplementedError("填入调试后的实现")

In [ ]:
assert window_sums([2, 7, 4, 1], 2) == [9, 11, 5]
assert window_sums([2, 7, 4, 1], 4) == [14]
assert window_sums([2, 7, 4, 1], 1) == [2, 7, 4, 1]
assert window_sums([2, 7, 4, 1], 5) == []
assert window_sums([], 1) == []
assert window_sums([5, -1, 1], 2) == [4, 0]
print("场景三：全部检查通过")

<details>
<summary>参考解答（点击展开）</summary>

```python
def window_sums(values, width):
    totals = []
    for start in range(len(values) - width + 1):
        window = values[start : start + width]
        totals.append(sum(window))
    return totals
```

#### 错误原因

循环只停在 `start = 0`、`1`。`range(2)` 不包含 `2`，最后一个合法起点被漏掉了。

#### 修正前后的本质差别

`len(values) - width` 是最后一个合法起点。把它加一后交给 `range` 作为不包含的终点，循环就会走到最后一个窗口。

</details>

## 场景四｜迟到的旧记录怎么覆盖了新记录？

### 场景与需求

测量设备上传记录时可能乱序。每条记录写成 `(设备编号, 采集时刻, 读数)`。对每个编号只保留采集时刻最大的记录。同一设备的记录时刻互不相同。

### 输入与输出

```python
[("A", 2, 20), ("B", 1, 7), ("A", 1, 10), ("B", 3, 9)]
# 预期：{"A": (2, 20), "B": (3, 9)}
```

A 的旧读数虽然第三条才收到，采集时刻仍然是 `1`。交换记录的到达顺序，保留下来的时刻和读数应一样。

### 你的任务

修复 `latest_by_id(rows)`，把记录列表整理成 `{设备编号: (采集时刻, 读数)}`。观察一个编号第二次出现前后，检查它原来的记录和本次收到的记录。哪次更新最早偏离预期？

In [ ]:
q4_rows = [("A", 2, 20), ("B", 1, 7), ("A", 1, 10), ("B", 3, 9)]
q4_expected = {"A": (2, 20), "B": (3, 9)}
print("到达顺序", q4_rows, "预期", q4_expected)

In [ ]:
def latest_by_id_trial(rows):
    latest = {}
    for device, time, reading in rows:
        previous = latest.get(device)
        if previous is None or time < previous[0]:
            latest[device] = (time, reading)
    return latest


q4_actual = latest_by_id_trial(q4_rows)
print("实际保留的记录", q4_actual)
assert q4_actual == q4_expected, "同一设备只保留时刻最大的记录"

In [ ]:
def latest_by_id(rows):
    raise NotImplementedError("填入调试后的实现")

In [ ]:
assert latest_by_id(q4_rows) == q4_expected
assert latest_by_id(list(reversed(q4_rows))) == q4_expected
assert latest_by_id([("X", 0, 10), ("X", 1, -5)]) == {"X": (1, -5)}
assert latest_by_id([("X", -1, 4), ("Y", -1, 4)]) == {"X": (-1, 4), "Y": (-1, 4)}
assert latest_by_id([]) == {}
print("场景四：全部检查通过")

<details>
<summary>参考解答（点击展开）</summary>

```python
def latest_by_id(rows):
    latest = {}
    for device, time, reading in rows:
        previous = latest.get(device)
        if previous is None or time > previous[0]:
            latest[device] = (time, reading)
    return latest
```

#### 错误原因

第三条是 A 的旧记录。此时 `time = 1`、`previous = (2, 20)`，条件 `1 < 2` 成立，旧读数覆盖了新读数。

#### 修正前后的本质差别

第一次看到编号仍直接保存。再次看到时，只在本次时刻大于已保存时刻时更新。比较的是采集时间，和读数的大小无关。

</details>

## 场景五｜步长已经减半，试点怎么越走越远？

### 场景与需求

想从给定起点找到一个让 `f(x) = (x - 3) ** 2` 变小的位置。方向已经给定。每次都从同一个起点计算 `试点 = 起点 + 步长 × 方向`。步长从 `2` 开始，试点未让 `f` 严格变小就减半，最多尝试四次。第一次满足条件就接受。

### 输入与输出

- 起点 `0`、方向 `6` → `{"trials": [12, 6, 3], "accepted": 3}`。试到 `6` 时函数值与起点相等，继续减半。
- 起点 `4`、方向 `-2` → `{"trials": [0, 2, 3], "accepted": 3}`。
- 四次都失败时，保留四个试点并将 `accepted` 设为 `None`。

### 你的任务

修复 `try_descent(start, direction)`，返回试点轨迹与接受的位置。重点看第二次试步，区分步长有没有减半，以及试点究竟从哪里出发。

In [ ]:
q5_start = 0
q5_direction = 6
q5_expected = {"trials": [12, 6, 3], "accepted": 3}
print("起点", q5_start, "方向", q5_direction, "预期", q5_expected)

In [ ]:
def try_descent_trial(start, direction):
    initial_value = (start - 3) ** 2
    trial = start
    step = 2
    trials = []
    for attempt in range(1, 5):
        trial = trial + step * direction
        trials.append(trial)
        trial_value = (trial - 3) ** 2
        if trial_value < initial_value:
            return {"trials": trials, "accepted": trial}
        step /= 2
    return {"trials": trials, "accepted": None}


q5_actual = try_descent_trial(q5_start, q5_direction)
print("实际轨迹", q5_actual)
assert q5_actual == q5_expected, "试点轨迹或接受的位置不符"

In [ ]:
def try_descent(start, direction):
    raise NotImplementedError("填入调试后的实现")

In [ ]:
assert try_descent(0, 6) == {"trials": [12, 6, 3], "accepted": 3}
assert try_descent(4, -2) == {"trials": [0, 2, 3], "accepted": 3}
assert try_descent(0, 1) == {"trials": [2], "accepted": 2}
assert try_descent(3, 2) == {"trials": [7, 5, 4, 3.5], "accepted": None}
print("场景五：全部检查通过")

<details>
<summary>参考解答（点击展开）</summary>

```python
def try_descent(start, direction):
    initial_value = (start - 3) ** 2
    step = 2
    trials = []
    for attempt in range(1, 5):
        trial = start + step * direction
        trials.append(trial)
        trial_value = (trial - 3) ** 2
        if trial_value < initial_value:
            return {"trials": trials, "accepted": trial}
        step /= 2
    return {"trials": trials, "accepted": None}
```

#### 错误原因

第二轮的 `step` 已经从 `2` 变成 `1`，但计算前的 `trial` 还是上一轮被拒绝的 `12`。所以新试点是 `12 + 1 * 6 = 18`。

#### 修正前后的本质差别

每轮试点都从固定的 `start` 计算，只有步长随尝试次数变化。被拒绝的值留在轨迹里供检查，不影响下一次出发的位置。

</details>

## 场景六｜组内均值修好了，整体均值怎么还是错的？

### 场景与需求

报告要同时给出每组的均值和所有有效观测的整体均值。每组先剔除 `None`，有效的 `0` 照常计入。没有有效观测的组保留组名、均值记为 `None`。所有组都没有有效观测时，整体均值也记为 `None`。

### 输入与输出

```python
{"A": [0, None, 6], "B": [12]}
# 预期：{"by_group": {"A": 3, "B": 12}, "overall": 6}
```

整体有三个观测 `0`、`6`、`12`，每个观测在整体均值中占相同权重。

### 你的任务

修复 `summarize_groups(groups)`，返回 `by_group` 和 `overall`。这里有两处独立的问题。用下面两份输入分开观察：只有 A 组时看 `0` 有没有被留下；另一份输入没有 `0`，但两组大小不同，用它单独检查整体均值。每改一处，重跑这两份输入，记录哪项通过了、哪项仍然失败。

In [ ]:
q6_cases = [
    (
        "检查有效的 0",
        {"A": [0, None, 6]},
        {"by_group": {"A": 3}, "overall": 3},
    ),
    (
        "检查不同组大小",
        {"A": [2, 4, 6], "B": [12]},
        {"by_group": {"A": 4, "B": 12}, "overall": 6},
    ),
    (
        "同时检查两处",
        {"A": [0, None, 6], "B": [12]},
        {"by_group": {"A": 3, "B": 12}, "overall": 6},
    ),
]
for label, groups, expected in q6_cases:
    print(label, groups, "预期", expected)

In [ ]:
def summarize_groups_trial(groups):
    by_group = {}
    for group, values in groups.items():
        observed = [value for value in values if value]
        by_group[group] = sum(observed) / len(observed) if observed else None
    group_means = [value for value in by_group.values() if value is not None]
    overall = sum(group_means) / len(group_means) if group_means else None
    return {"by_group": by_group, "overall": overall}


q6_failed = []
for label, groups, expected in q6_cases:
    actual = summarize_groups_trial(groups)
    print(label, "通过" if actual == expected else "失败", "实际", actual)
    if actual != expected:
        q6_failed.append(label)
assert not q6_failed, f"未通过的用例：{q6_failed}"

In [ ]:
def summarize_groups(groups):
    raise NotImplementedError("填入调试后的实现")

In [ ]:
for label, groups, expected in q6_cases:
    assert summarize_groups(groups) == expected, label

assert summarize_groups({"A": [None], "B": [0, 4], "C": []}) == {
    "by_group": {"A": None, "B": 2, "C": None}, "overall": 2,
}
assert summarize_groups({"A": [None], "B": []}) == {
    "by_group": {"A": None, "B": None}, "overall": None,
}
assert summarize_groups({}) == {"by_group": {}, "overall": None}
print("场景六：全部检查通过")

<details>
<summary>参考解答（点击展开）</summary>

```python
def summarize_groups(groups):
    by_group = {}
    all_observed = []
    for group, values in groups.items():
        observed = [value for value in values if value is not None]
        by_group[group] = sum(observed) / len(observed) if observed else None
        all_observed.extend(observed)
    overall = sum(all_observed) / len(all_observed) if all_observed else None
    return {"by_group": by_group, "overall": overall}
```

#### 错误原因

只有 A 组的输入里，筛选后原本有 `[0, 6]`，实际只剩 `[6]`。没有 `0` 的输入里，两组均值已经正确，整体却把 `4` 和 `12` 直接平均成 `8`。这时 A 组三个观测、B 组一个观测，直接平均给两组各占一半权重。

#### 修正前后的本质差别

第一处只排除 `None`，让组内计算读到全部有效值。第二处让整体均值使用所有有效观测，组内均值仍然逐组计算。只修筛选时，组合用例仍会报整体均值 `7.5`；两处都修好才会得到 `6`。

</details>

### 场景六 · 选做｜构造一组能区分两种算法的数据

下方两种算法只接收各组非空、且全是有效数字的数据。给定示例里，两种算法都得到 `6`。

换一组你自己的数据，让两份代码得出不同结果。先算出预期的整体均值，再用断点检查两份实现各自拿了哪些数来求均值。

In [ ]:
def overall_from_groups(groups):
    means = [sum(values) / len(values) for values in groups.values()]
    return sum(means) / len(means)


def overall_from_observations(groups):
    observations = [value for values in groups.values() for value in values]
    return sum(observations) / len(observations)


# 两种算法对这份输入结果相同。替换为你自己的数据，再运行本格。
q6_extension_groups = {"A": [2, 4], "B": [8, 10]}
print("先算组均值", overall_from_groups(q6_extension_groups))
print("直接用所有观测", overall_from_observations(q6_extension_groups))